In [ ]:
import socket

HOST = 'localhost'
PORT = 80         # Puerto elegido "al azar"

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)  # SOCK_DGRAM
s.connect((HOST, PORT))   # El servidor hace connect()
conn, addr = s.accept()

while True:
    data = conn.recv(4096000)   # Buffer de 4 MB
    conn.sendall(data)
    # No hay condición de cierre


OSError: [Errno 95] Operation not supported

#Vamos paso a paso

In [ ]:
# Con el siguiente codigo:

#import socket

#HOST = 'localhost'
#PORT = 80         # Puerto elegido "al azar"

#s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)  # SOCK_DGRAM
#s.connect((HOST, PORT))   # El servidor hace connect()
#conn, addr = s.accept()

#while True:
#    data = conn.recv(4096000)   # Buffer de 4 MB
#    conn.sendall(data)
    # No hay condición de cierre

# Nos salio el siguiente error:

#---------------------------------------------------------------------------
#OSError                                   Traceback (most recent call last)
#/tmp/ipykernel_1529/2931581199.py in <cell line: 0>()
#      6 s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)  # SOCK_DGRAM
#      7 s.connect((HOST, PORT))   # El servidor hace connect()
#      9
#     10 while True:
#
#/usr/lib/python3.12/socket.py in accept(self)
#    293         For IP sockets, the address info is a pair (hostaddr, port).
#    294         """
#--> 295         fd, addr = self._accept()
#    296         sock = socket(self.family, self.type, self.proto, fileno=fd)
#    297         # Issue #7995: if no default timeout is set and the listening
#
#OSError: [Errno 95] Operation not supported

# Buscando en fuentes una explicacion del error se dio lo siguiente:
# El error salta específicamente porque en la línea de s.accept() estamos
# intentando aceptar una conexión en un socket configurado como UDP (SOCK_DGRAM).
# El protocolo UDP es "sin conexión" (solo lanza y recibe paquetes sueltos),
# por lo que pedirle que "acepte" o mantenga una conexión no tiene sentido y el
# sistema lanza el error diciendo que esa operación "no está soportada".


# |---------------------------------------------------------------------------|
# Lo que debemos hacer es:

# Identificar todos los errores conceptuales y/o de implementación (hay al
# menos 3) y explica cómo corregirlos:

# Para eso vamos a analizar el codigo paso a paso:

# En la linea 1 tenemos: "import socket" el cual sirve para:
# para acceder a la biblioteca estándar que permite la comunicación de red,
# facilitando la creación de conexiones entre clientes y servidores.

# Luego de eso tenemos: (HOST = 'localhost').
# para indicar a un programa, servidor web o base de datos que se ejecute o
# conecte únicamente en tu propia computadora.

# En base a eso tenemos el port, (PORT = 80         # Puerto elegido "al azar").
# El cual sirve para indicarle a tu programa que utilice el puerto estándar
# para el tráfico web HTTP.

# Luego tenemos la línea: s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
# Sirve para crear el objeto "socket" especificando la familia de direcciones
# (IPv4 con AF_INET).
# Hay que estar atentos ya que puede ser que por estas lineas este el error

# En la siguiente línea tenemos: s.connect((HOST, PORT))
# Sirve para intentar establecer una conexión activa hacia un servidor remoto.
# SEGUNDO ERROR AQUI !!!!!!!!!!!!
# ERROR CONCEPTUAL: Un servidor NUNCA hace "connect()", eso lo hace el cliente
# para ir a buscar al servidor. El servidor debería usar "bind()" para adueñarse
# del puerto y quedarse esperando.

# Luego tenemos: conn, addr = s.accept()
# Sirve para bloquear la ejecución y esperar pasivamente una conexión
# entrante. Cuando un cliente se conecta, devuelve un nuevo socket (conn)
# dedicado a esa comunicación y la dirección del cliente (addr).
# TERCER ERROR Y EL QUE DAÑA TODO !!!!!!!!!!!!
# ERROR DE IMPLEMENTACION: Como dijimos arriba, accept() es exclusivo para TCP
# (SOCK_STREAM). No se puede usar con UDP. Para corregirlo, si el código debía
# ser UDP, hay que borrar esta línea. Si debía ser TCP, había que cambiar la
# configuración del socket arriba a SOCK_STREAM y agregar s.listen().

# A continuación tenemos: while True:
# este nos sirve para iniciar un bucle infinito que permite que el servidor
# se mantenga encendido y procesando múltiples flujos de información
# continuamente sin detenerse tras recibir el primer mensaje.

# Dentro del bucle tenemos: data = conn.recv(4096000)
# Sirve para leer y recibir los datos que el cliente ha enviado a través
# del canal, especificando la cantidad máxima de bytes que puede leer a la vez en memoria.
# PRIMER ERROR DE PRIMERAS !!!!!!!!!!!!
# Aqui tenemos un ERROR CONCEPTUAL (Búfer) ya que se asigna un búfer de 4 MB
# de golpe es ineficiente y consume demasiada memoria sin necesidad.
# Lo estándar y recomendado segun fuentes es usar tamaños de búfer más pequeños,
# como 1024 o 4096 bytes. Además, para UDP se debería usar recvfrom(), no recv().

# Finalmente tenemos: conn.sendall(data)
# Sirve para enviar de vuelta (eco) toda la información recibida (data)
# hacia el cliente, asegurando que todos los bytes sean transmitidos
# por el canal de red
# CUARTO ERROR PARA REMATAR !!!!!!!!!!!!
# ERROR DE IMPLEMENTACION: "sendall()" también es una función exclusiva de TCP.
# Para UDP se debería usar "sendto(data, addr)" para devolver el mensaje.

# XD

#

# CORRECCIONES DEL CODIGO


In [ ]:
import socket

HOST = 'localhost'
PORT = 8080        # Cambiado a 8080 para no tener problemas de permisos del SO

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)  # SOCK_DGRAM (UDP)
s.bind((HOST, PORT))   # CORREGIDO: El servidor hace bind(), no connect()

print(f"Servidor UDP escuchando en el puerto {PORT}...")

while True:
    # CORREGIDO: Usamos recvfrom() para UDP y un buffer estándar de 1024
    data, addr = s.recvfrom(1024)

    # CORREGIDO: Usamos sendto() para devolver el mensaje a la dirección origen
    s.sendto(data, addr)

Servidor UDP escuchando en el puerto 8080...


KeyboardInterrupt: 

#Como se observa, no hay errores al momento de ejecutar. Mision cumplida xd


# Conclusiones:

## ERROR 1: El tamaño del búfer y el método de lectura
Teníamos asignado un búfer gigante de 4 MB (`4096000`), lo cual es ineficiente, consume memoria en vano y supera por mucho el límite físico de un paquete UDP. Además, usábamos `recv()`, que es exclusivo de TCP.
* **Solución:** Cambiar el tamaño a algo estándar y seguro como `1024` bytes y usar la función correcta para leer en UDP: `recvfrom()`.

## ERROR 2: Confusión en el rol del servidor
Estábamos usando `s.connect()`. Conceptualmente esto está mal porque un servidor no "sale" a conectarse con nadie; el servidor debe adueñarse de un puerto y quedarse escuchando a que los clientes lleguen.
* **Solución:** Reemplazar `connect()` por `bind()` para que el servidor se ancle a la dirección y puerto asignados.

## ERROR 3: Mezclar UDP con TCP (El causante del crash)
Usamos la función `accept()` en un socket configurado como UDP (`SOCK_DGRAM`). Como UDP es un protocolo que simplemente lanza y recibe datos "sin conexión" fija, pedirle al sistema que "acepte" una conexión hace que todo explote y lance el error `Operation not supported`.
* **Solución:** Eliminar completamente la línea de `accept()`. En UDP no hace falta, pasamos directamente a escuchar los mensajes.

## ERROR 4: El método de envío incorrecto
Al intentar devolver el mensaje como un eco, el código usaba `sendall()`, una función diseñada para asegurar la transmisión en conexiones TCP.
* **Solución:** Cambiarlo por `sendto()`, que es la función correcta en UDP para agarrar los datos y mandarlos a la dirección específica (`addr`) que nos habló originalmente.
